# AutoEconSentiment Transformer Walkthrough

This notebook shows how to use the optional transformer configuration path added to `auto-econ-sentiment`.

The first sections run without downloading any Hugging Face model. They demonstrate:

- the original `Econ_Text_Algos`-style model list,
- conversion from `label_mapping` and `sentiment_values` to the package's internal `label_map`,
- sentence-level aggregation by `id_text`,
- harmonized positive/neutral/negative counts, shares, and net sentiment.

The final section shows the optional real-model workflow for environments installed with `uv sync --extra transformers`.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

REPO_ROOT

In [ ]:
import pandas as pd

from auto_econ_sentiment.pipeline import AutoEconSentiment
from auto_econ_sentiment.models.sentiment_transformers import SentimentTransformers

## 1. Original-style transformer config

The transformer config can use the older `Econ_Text_Algos` model-list style. Each model maps raw model labels to semantic labels with `label_mapping`, then maps semantic labels to numeric sentiment direction with `sentiment_values`.

In [ ]:
transformer_config = {
    "enabled": True,
    "text_column_transformer": "text_clean",
    "aggregation_methods": ["sentence_pos"],
    "output_schema": "shares",
    "net_sentiment_formula": "positive_minus_negative",
    "models": [
        {
            "name": "gtfintechlab/FOMC-RoBERTa",
            "short_name": "fomc",
            "num_labels": 3,
            "max_length": 512,
            "batch_size": 8,
            "label_mapping": {
                "LABEL_0": "positive",
                "LABEL_1": "negative",
                "LABEL_2": "neutral",
            },
            "sentiment_values": {
                "positive": 1,
                "negative": -1,
                "neutral": 0,
            },
        },
        {
            "name": "ProsusAI/finbert",
            "short_name": "finbertpro",
            "num_labels": 3,
            "max_length": 512,
            "batch_size": 8,
            "label_mapping": {
                "neutral": "neutral",
                "positive": "positive",
                "negative": "negative",
            },
            "sentiment_values": {
                "neutral": 0,
                "positive": 1,
                "negative": -1,
            },
        },
    ],
}

expanded = AutoEconSentiment._expand_transformer_model_configs(
    transformer_config=transformer_config,
    default_text_column="text_clean",
)
pd.DataFrame([
    {
        "model_name": config["model_name"],
        "model_name_short": config["model_name_short"],
        "aggregation": config["aggregation"],
        "output_schema": config.get("output_schema"),
        "label_map": config["label_map"],
    }
    for config in expanded
])

## 2. Sample sentence-level data

Sentence aggregation expects repeated `id_text` values. Each sentence row is scored separately, then grouped back to the source document.

In [ ]:
sample_sentences = pd.DataFrame(
    {
        "id_text": [1, 1, 1, 2, 2],
        "text_clean": [
            "Bank capital remains resilient and market functioning is strong.",
            "Credit stress increased in vulnerable sectors.",
            "The report reviews recent market developments.",
            "Inflation progress supports a stronger outlook.",
            "Funding conditions remain broadly stable.",
        ],
    }
)
sample_sentences

## 3. A no-download demo classifier

This tiny class uses the real `SentimentTransformers` post-processing and sentence aggregation methods, but replaces Hugging Face inference with deterministic keyword probabilities. It is useful for understanding output structure without installing `torch`.

In [ ]:
class DemoConfig:
    id2label = {0: "LABEL_0", 1: "LABEL_1", 2: "LABEL_2"}


class DemoModel:
    config = DemoConfig()


class KeywordTransformer(SentimentTransformers):
    def __init__(self, df_input: pd.DataFrame, text_column: str = "text_clean"):
        self.input_df = df_input
        self.text_column = text_column
        self.model_name = "keyword-sample"
        self.model_name_short = "sample"
        self.label_map = {"LABEL_0": 1.0, "LABEL_1": -1.0, "LABEL_2": 0.0}
        self.output_schema = "shares"
        self.net_sentiment_formula = "positive_minus_negative"
        self.model = DemoModel()
        self.num_labels = 3
        self.df_labels = None
        self.df_sentence_probabilities = None
        self.df_sentiment_output = None

    def analyze_sentiment_single(self, texts, return_probabilities=True):
        probabilities = []
        for text in texts:
            text_lower = str(text).lower()
            if any(word in text_lower for word in ("strong", "progress", "resilient")):
                probabilities.append([0.9, 0.05, 0.05])
            elif any(word in text_lower for word in ("weak", "recession", "stress")):
                probabilities.append([0.05, 0.9, 0.05])
            else:
                probabilities.append([0.05, 0.05, 0.9])
        return self._postprocess_predictions(probabilities, return_probabilities=return_probabilities)


demo_transformer = KeywordTransformer(sample_sentences)
scores, sentence_probabilities = demo_transformer.sentiment_pipeline(
    aggregation="bysentence",
    sentence_probability_cutoff=0.7,
)

In [ ]:
scores[
    [
        "sample_count_positive",
        "sample_count_neutral",
        "sample_count_negative",
        "sample_share_positive",
        "sample_share_neutral",
        "sample_share_negative",
        "sample_net_sentiment",
        "sample_sentiment_bysentence",
    ]
]

In [ ]:
sentence_probabilities

## 4. Optional real-model run

Use this section only after installing transformer dependencies:

```bash
uv sync --extra transformers
```

The model will be downloaded from Hugging Face unless it is already cached locally.

In [ ]:
# Uncomment to run with a real Hugging Face model.
#
# real_transformer = SentimentTransformers(
#     df_input=sample_sentences,
#     text_column="text_clean",
#     model_name="gtfintechlab/FOMC-RoBERTa",
#     model_name_short="fomc",
#     num_labels=3,
#     max_length=512,
#     batch_size=4,
#     output_schema="shares",
#     net_sentiment_formula="positive_minus_negative",
#     label_map={"LABEL_0": 1, "LABEL_1": -1, "LABEL_2": 0},
# )
# real_scores, real_sentence_probabilities = real_transformer.sentiment_pipeline(
#     aggregation="bysentence",
#     sentence_probability_cutoff=0.7,
# )
# real_scores